# 02 - Model Training & Validation Orchestration

**Competition:** Solar Filament Segmentation Challenge 2026  
**Stage:** Baseline Training (U-Net with ResNet34 backbone)  
**Checkpoint Evaluation:** Safe checkpoint-loading and diagnostic evaluation without retraining.

> **Principle:** Kaggle notebooks import modular Python components from `src/filament_seg/`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'dataset.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'dataset.py').is_file(), cwd
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import set_seed, load_config, get_device, get_logger
from filament_seg.model import build_model
from filament_seg.losses import BCEDiceLoss
from filament_seg.evaluate import compute_dice, compute_iou, evaluate_semantic_predictions
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.preprocessing import ForegroundAwareCrop, get_train_transform, get_val_transform
from filament_seg.train import load_checkpoint, validate_one_epoch

config = load_config(project_root / 'configs' / 'baseline.yaml')
set_seed(config['project']['seed'])
device = get_device(config['training'].get('device'))
print(f'Python: {sys.executable}')
print(f'Project root: {project_root}')
print(f'Device selected: {device}')
print(f'Experiment: {config["project"]["experiment_name"]}')

## 1. Dataset and Image-Level Split

Split 707 physical filenames once (seed 42, 80/20) so alternative annotation sets never cross validation boundaries.

In [ ]:
train_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'train'
image_dir = train_dir / 'train_images'
annotation_path = train_dir / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'

train_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_train_transform(512))
val_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_val_transform(512))
assert len(train_dataset) == len(val_dataset) == 707

indices = np.random.default_rng(config['project']['seed']).permutation(len(train_dataset))
train_indices = indices[:565].tolist()
val_indices = indices[565:].tolist()

train_names = {train_dataset.filenames[i] for i in train_indices}
val_names = {val_dataset.filenames[i] for i in val_indices}
assert len(train_names) == 565 and len(val_names) == 142
assert train_names.isdisjoint(val_names)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)
print(f'Physical images: {len(train_dataset)}; Train: {len(train_subset)}; Validation: {len(val_subset)}')

## 2. Preprocessing, DataLoaders, and Spatial Verification

In [ ]:
train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=0)
print(f'Train DataLoader: {len(train_loader)} batches; Val DataLoader: {len(val_loader)} batches')

batch = next(iter(train_loader))
print(f'Sample batch image shape: {tuple(batch["image"].shape)}, range: [{batch["image"].min():.2f}, {batch["image"].max():.2f}]')

## 3. Baseline Model & Loss Configuration

Instantiate U-Net with ResNet34 encoder and BCEDiceLoss (pos_weight=30.0).

In [ ]:
model = build_model(pretrained=False).to(device)
criterion = BCEDiceLoss(pos_weight=config['training']['loss']['pos_weight'])
print('U-Net with ResNet34 encoder initialized.')

## 4. Checkpoint Loading & Training Diagnostics

Load `checkpoints/unet_resnet34/best_model.pth` and display training and validation history across epochs.

In [ ]:
ckpt_path = project_root / 'checkpoints' / 'unet_resnet34' / 'best_model.pth'
ckpt = load_checkpoint(ckpt_path, model, map_location=device)
print(f'Loaded checkpoint from epoch: {ckpt.get("epoch")}, Best Val Dice: {ckpt.get("best_val_dice"):.4f}')

history = ckpt.get('history', [])
if history:
    epochs = [h['epoch'] for h in history]
    val_dices = [h['val_dice'] for h in history]
    val_ious = [h['val_iou'] for h in history]
    train_losses = [h['train_loss'] for h in history]
    val_losses = [h['val_loss'] for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), dpi=120)
    ax1.plot(epochs, train_losses, label='Train Loss', marker='o')
    ax1.plot(epochs, val_losses, label='Val Loss', marker='s')
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss History')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, val_dices, label='Val Dice', marker='^', color='green')
    ax2.plot(epochs, val_ious, label='Val IoU', marker='v', color='orange')
    ax2.axvline(11, color='red', linestyle='--', label='Best Checkpoint (Epoch 11)')
    ax2.set(xlabel='Epoch', ylabel='Score', title='Validation Metrics')
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Validation Metric Diagnostics

Evaluate model on validation loader patches to reproduce reported metrics.

In [ ]:
patch_metrics = validate_one_epoch(model, val_loader, criterion, device=device)
print('Center-crop Patch Validation Results:')
print(f'  Val Loss: {patch_metrics["val_loss"]:.4f}')
print(f'  Val Dice: {patch_metrics["val_dice"]:.4f}')
print(f'  Val IoU:  {patch_metrics["val_iou"]:.4f}')
print('Note: Patch validation center-crop is an auxiliary check; primary full-resolution evaluation is executed in 03_submission.ipynb.')